# Test a video: LCD + speed + RPM, using the Drive API

Run the cells individually. Start with **Hardware accelerator: None**.

This notebook uses the existing LCD epoch 9, speed epoch 11 and RPM epoch 9 checkpoints. It preserves the previous diagnostic settings, full input/output labels and larger RPM text. It does not train or improve those models. Raw readings remain unvalidated; known tick-position and RPM-crop errors remain.

**One-time setup:** in Colab's key/Secrets panel, add the private OAuth JSON under **TRIUMPH_DRIVE_OAUTH**, then allow this notebook to access that secret. Never paste credentials into a code cell or notebook output. Use the configured Google account. The secret is read only in memory; the public notebook contains none of its values. An External/Testing OAuth grant expires after about seven days and needs renewal; this is not permanent authorization.

The first CPU check verifies account access and the migrated model package. To browse for a video, turn **SETUP_CHECK_ONLY** off and rerun cell 1. The upload is saved through the Drive API before you switch to T4. Cell 2 downloads verified assets, runs the unchanged models, saves all results privately, then releases the GPU. There is no Drive mount.


In [ ]:
#@title 1. CPU setup check / browse and save a video
SETUP_CHECK_ONLY = True #@param {type:"boolean"}
START_SECONDS = 0.0 #@param {type:"number"}
MAX_SECONDS = 0.0 #@param {type:"number"}
# MAX_SECONDS=0 processes the entire remainder. Set 4 for a short experiment.
# This public notebook contains no credentials, weights, or private file IDs.
from pathlib import Path
import hashlib, importlib.util, os, sys, urllib.request, uuid
CODE_COMMIT = '8cf85692aee5ab87e3a62b353977d20c7e1bb315'
EXPECTED_FILES = {'drive_api_store.py': {'path': 'learned_lcd_v1/drive_api_video_v1/drive_api_store.py', 'bytes': 22670, 'sha256': '681f372270c8b42ff6d1bf7f3f3164113481fa118924fb542404033bd58596f4'}, 'api_colab_workflow.py': {'path': 'learned_lcd_v1/drive_api_video_v1/api_colab_workflow.py', 'bytes': 22882, 'sha256': 'dc18c493a9d89bbe1c3484b3e5d9857e7bdb2c7af4a13eb7056bf59dc5e90966'}, 'colab_video_helpers.py': {'path': 'learned_lcd_v1/upload_video_test_v1/colab_video_helpers.py', 'bytes': 41443, 'sha256': 'f48f161fcf9ed2c1c86754698f41554090d3fb9ea671c693322819be4e298bfe'}, 'evaluate_uploaded_video.py': {'path': 'learned_lcd_v1/upload_video_test_v1/evaluate_uploaded_video.py', 'bytes': 28779, 'sha256': '5ca4b7b1c59be091b23f999bc8e571fb3275eea37792abb67b3e147b11cb33ef'}, 'evaluate_lcd_speed_rpm_batched.py': {'path': 'learned_lcd_v1/upload_video_test_v1/evaluate_lcd_speed_rpm_batched.py', 'bytes': 55284, 'sha256': '0c5df9a13e5cd904d76339a74de8d84675965041ee166eacf94e933214947df3'}, 'benchmark_lcd_positioning.py': {'path': 'learned_lcd_v1/upload_video_test_v1/benchmark_lcd_positioning.py', 'bytes': 21561, 'sha256': '68c299ada88ff433fea7e57ac379823ee9d6848d61f0605556642ecc3c5957e0'}}
RUNTIME_DIR = Path('/content/triumph_api_runtime') / CODE_COMMIT
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
for name, metadata in EXPECTED_FILES.items():
    target = RUNTIME_DIR / name
    if target.is_file():
        data = target.read_bytes()
    else:
        url = 'https://raw.githubusercontent.com/sainadh812/triumph-speedometer-dataset/' + CODE_COMMIT + '/' + metadata['path']
        try:
            with urllib.request.urlopen(url, timeout=60) as response:
                data = response.read()
        except Exception:
            raise RuntimeError('Could not download reviewed public code: ' + name) from None
    if len(data) != metadata['bytes'] or hashlib.sha256(data).hexdigest() != metadata['sha256']:
        raise RuntimeError('Public runtime checksum mismatch: ' + name)
    if not target.exists():
        temporary = target.with_name(name + '.' + uuid.uuid4().hex + '.tmp')
        temporary.write_bytes(data)
        os.replace(temporary, target)
for name in ('drive_api_store', 'api_colab_workflow'):
    specification = importlib.util.spec_from_file_location(name, RUNTIME_DIR / (name + '.py'))
    module = importlib.util.module_from_spec(specification)
    sys.modules[name] = module
    specification.loader.exec_module(module)
workflow = sys.modules['api_colab_workflow']

if SETUP_CHECK_ONLY:
    setup_result = workflow.preflight(RUNTIME_DIR)
    print("To browse for a video, turn SETUP_CHECK_ONLY off and rerun this cell.")
else:
    job = workflow.upload_video(RUNTIME_DIR, START_SECONDS, MAX_SECONDS)


In [ ]:
#@title 2. Switch to T4, then process the saved video
# Blank JOB_FOLDER_ID selects the newest completed input upload.
# An already completed job is displayed without running inference again.
JOB_FOLDER_ID = "" #@param {type:"string"}
AUTO_DISCONNECT = True #@param {type:"boolean"}
workflow_started = False
try:
    # This public notebook contains no credentials, weights, or private file IDs.
    from pathlib import Path
    import hashlib, importlib.util, os, sys, urllib.request, uuid
    CODE_COMMIT = '8cf85692aee5ab87e3a62b353977d20c7e1bb315'
    EXPECTED_FILES = {'drive_api_store.py': {'path': 'learned_lcd_v1/drive_api_video_v1/drive_api_store.py', 'bytes': 22670, 'sha256': '681f372270c8b42ff6d1bf7f3f3164113481fa118924fb542404033bd58596f4'}, 'api_colab_workflow.py': {'path': 'learned_lcd_v1/drive_api_video_v1/api_colab_workflow.py', 'bytes': 22882, 'sha256': 'dc18c493a9d89bbe1c3484b3e5d9857e7bdb2c7af4a13eb7056bf59dc5e90966'}, 'colab_video_helpers.py': {'path': 'learned_lcd_v1/upload_video_test_v1/colab_video_helpers.py', 'bytes': 41443, 'sha256': 'f48f161fcf9ed2c1c86754698f41554090d3fb9ea671c693322819be4e298bfe'}, 'evaluate_uploaded_video.py': {'path': 'learned_lcd_v1/upload_video_test_v1/evaluate_uploaded_video.py', 'bytes': 28779, 'sha256': '5ca4b7b1c59be091b23f999bc8e571fb3275eea37792abb67b3e147b11cb33ef'}, 'evaluate_lcd_speed_rpm_batched.py': {'path': 'learned_lcd_v1/upload_video_test_v1/evaluate_lcd_speed_rpm_batched.py', 'bytes': 55284, 'sha256': '0c5df9a13e5cd904d76339a74de8d84675965041ee166eacf94e933214947df3'}, 'benchmark_lcd_positioning.py': {'path': 'learned_lcd_v1/upload_video_test_v1/benchmark_lcd_positioning.py', 'bytes': 21561, 'sha256': '68c299ada88ff433fea7e57ac379823ee9d6848d61f0605556642ecc3c5957e0'}}
    RUNTIME_DIR = Path('/content/triumph_api_runtime') / CODE_COMMIT
    RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
    for name, metadata in EXPECTED_FILES.items():
        target = RUNTIME_DIR / name
        if target.is_file():
            data = target.read_bytes()
        else:
            url = 'https://raw.githubusercontent.com/sainadh812/triumph-speedometer-dataset/' + CODE_COMMIT + '/' + metadata['path']
            try:
                with urllib.request.urlopen(url, timeout=60) as response:
                    data = response.read()
            except Exception:
                raise RuntimeError('Could not download reviewed public code: ' + name) from None
        if len(data) != metadata['bytes'] or hashlib.sha256(data).hexdigest() != metadata['sha256']:
            raise RuntimeError('Public runtime checksum mismatch: ' + name)
        if not target.exists():
            temporary = target.with_name(name + '.' + uuid.uuid4().hex + '.tmp')
            temporary.write_bytes(data)
            os.replace(temporary, target)
    for name in ('drive_api_store', 'api_colab_workflow'):
        specification = importlib.util.spec_from_file_location(name, RUNTIME_DIR / (name + '.py'))
        module = importlib.util.module_from_spec(specification)
        sys.modules[name] = module
        specification.loader.exec_module(module)
    workflow = sys.modules['api_colab_workflow']

    workflow_started = True
    receipt = workflow.run_video(RUNTIME_DIR, job_folder_id=JOB_FOLDER_ID, auto_disconnect=AUTO_DISCONNECT)
except BaseException:
    if not workflow_started and AUTO_DISCONNECT:
        print("Public code setup failed before any inference. Releasing this runtime.")
        try:
            from google.colab import runtime
            runtime.unassign()
        except Exception:
            print("Use Runtime > Disconnect and delete runtime if still connected.")
    raise


### 3. Download the labeled video

Wait for **Drive API backup verified** in cell 2. Open **Open or download labeled video**, then download the MP4 directly from Drive. The GPU can remain disconnected during that download. Predictions, timing reports and model provenance are stored alongside it.

Reader FPS measures model-pipeline processing; end-to-end FPS includes video decoding, drawing labels and encoding. Video playback FPS is a separate quantity. Drive upload time is additional.

If saving fails, runtime results are retained. Rerun cell 2 to retry an existing pending upload without repeating inference. Do not change or delete that runtime until the results are saved. If a failure happens before a recovery pointer can be written, the error prints the exact runtime folder to preserve manually.

Private files live under **My Drive / SpeedometerTraining_API_v1**. Existing folders and checkpoints from earlier accounts are preserved.
